# Airbnb Price Analysis — London (Sept 2025)

**What drives Airbnb prices in London, and how much does location actually matter?**

London is one of the world's most expensive cities to visit — but the price gap between a flat in Kensington and one in Barking is enormous. This notebook digs into the Inside Airbnb dataset to quantify exactly what's driving that gap.

We're using the [Inside Airbnb](https://insideairbnb.com) full listings dataset scraped in **September 2025** — ~80,000 listings, 79 features including bedrooms, review scores across 7 dimensions, estimated occupancy, host status, and full amenity lists.

---

**Roadmap:**
1. Load & clean the data (price strings, boolean flags, text fields)
2. Understand the price distribution
3. Room type and property type breakdown
4. How capacity (guests, bedrooms, beds) scales with price
5. Review scores — do ratings actually correlate with price?
6. Superhost and instant booking premiums
7. Location: Inner vs. Outer London → borough → neighbourhood
8. Geographic price maps
9. Amenity analysis — which features command a premium?
10. Random Forest: rank the drivers objectively

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import ast
import re
from collections import Counter
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder

plt.rcParams.update({
    'figure.figsize': (12, 5),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
})
sns.set_palette('husl')

print('Libraries loaded ✓')

## 1. Load the Data

In [ ]:
URL = 'https://data.insideairbnb.com/united-kingdom/england/london/2025-09-14/data/listings.csv.gz'

print(f'Fetching London listings from Inside Airbnb...')
df = pd.read_csv(URL, compression='infer', low_memory=False)
print(f'Loaded {len(df):,} listings × {df.shape[1]} columns')
df.head(3)

In [ ]:
# Quick overview: which columns have data?
missing = df.isnull().mean().sort_values(ascending=False)
print('Columns with >30% missing:')
print(missing[missing > 0.3].apply(lambda x: f'{x:.0%}').to_string())

## 2. Cleaning Up

The full Inside Airbnb dataset needs more preprocessing than a pre-cleaned Kaggle file — prices come as `"$149.00"`, booleans as `"t"/"f"`, and bathrooms are buried in text strings. Let's fix all that.

In [ ]:
# --- Price: strip currency symbol and convert to float ---
df['price'] = (
    df['price']
    .astype(str)
    .str.replace(r'[\$£,]', '', regex=True)
    .str.strip()
    .replace('nan', np.nan)
    .pipe(pd.to_numeric, errors='coerce')
)

# --- Boolean columns: 't'/'f' → True/False ---
for col in ['host_is_superhost', 'instant_bookable', 'host_identity_verified', 'host_has_profile_pic']:
    df[col] = df[col].map({'t': True, 'f': False})

# --- Bathrooms: extract number from text (e.g. '1.5 baths', '1 shared bath') ---
def parse_bathrooms(txt):
    if pd.isnull(txt):
        return np.nan
    txt = str(txt).lower()
    if 'half' in txt:
        return 0.5
    nums = re.findall(r'\d+\.?\d*', txt)
    return float(nums[0]) if nums else np.nan

df['bathrooms_num'] = df['bathrooms_text'].apply(parse_bathrooms)
df['bath_shared'] = df['bathrooms_text'].str.lower().str.contains('shared', na=False)

# --- Amenities: parse JSON-like list → count + individual flags ---
def parse_amenities(x):
    try:
        return ast.literal_eval(x) if pd.notnull(x) else []
    except:
        return []

df['amenities_list'] = df['amenities'].apply(parse_amenities)
df['n_amenities'] = df['amenities_list'].apply(len)

# --- Drop listings with no price ---
df = df.dropna(subset=['price'])
df = df[df['price'] > 0]

# --- Inner vs Outer London classification ---
inner_london = {
    'City of London', 'Camden', 'Greenwich', 'Hackney', 'Hammersmith and Fulham',
    'Islington', 'Kensington and Chelsea', 'Lambeth', 'Lewisham', 'Newham',
    'Southwark', 'Tower Hamlets', 'Wandsworth', 'Westminster', 'Haringey'
}
df['london_zone'] = df['neighbourhood_cleansed'].apply(
    lambda x: 'Inner London' if x in inner_london else 'Outer London'
)

print(f'Clean dataset: {len(df):,} listings')
print(f'Price range: £{df["price"].min():.0f} – £{df["price"].quantile(0.99):.0f} (99th pct)')
print(f'\nInner London: {(df["london_zone"]=="Inner London").sum():,}  |  Outer London: {(df["london_zone"]=="Outer London").sum():,}')

In [ ]:
# Plotting subset: cap at £500 to keep charts readable (~97% of data)
df_plot = df[df['price'] <= 500].copy()
print(f'Plotting subset: {len(df_plot):,} listings ({len(df_plot)/len(df):.1%} of total, price ≤ £500)')
print(f"\nPrice summary (full dataset):")
print(df['price'].describe().apply(lambda x: f'£{x:,.0f}'))

## 3. Price Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw
axes[0].hist(df_plot['price'], bins=70, color='royalblue', edgecolor='white', linewidth=0.3)
axes[0].axvline(df_plot['price'].median(), color='tomato', linestyle='--', lw=1.8, label=f'Median: £{df_plot["price"].median():.0f}')
axes[0].axvline(df_plot['price'].mean(), color='orange', linestyle='--', lw=1.8, label=f'Mean: £{df_plot["price"].mean():.0f}')
axes[0].set_title('Nightly Price — Raw')
axes[0].set_xlabel('Price per night (£)')
axes[0].set_ylabel('Listings')
axes[0].legend()

# Log
axes[1].hist(np.log1p(df_plot['price']), bins=70, color='mediumseagreen', edgecolor='white', linewidth=0.3)
axes[1].set_title('Nightly Price — Log Scale')
axes[1].set_xlabel('log(price + 1)')
axes[1].set_ylabel('Listings')

plt.suptitle('London Airbnb Nightly Prices — Sept 2025', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'Skewness: {df_plot["price"].skew():.2f}  |  Mean/Median ratio: {df_plot["price"].mean()/df_plot["price"].median():.2f}x')

Very similar shape to what we'd see in any major city — heavily right-skewed with a long luxury tail. The log-transformed version is near-normal, confirming that **price behaves multiplicatively** (a 20% premium, not a £20 premium). We'll model log-price in the Random Forest section.

## 4. Room Type & Property Type

In [ ]:
room_stats = (
    df.groupby('room_type')['price']
    .agg(count='count', median='median', mean='mean')
    .sort_values('median', ascending=False)
)
room_stats['share'] = (room_stats['count'] / room_stats['count'].sum() * 100).round(1)
print(room_stats.to_string())

In [ ]:
# Top property types (consolidate long tail into 'Other')
top_props = df['property_type'].value_counts().head(10).index
df_plot['prop_type_clean'] = df_plot['property_type'].where(df_plot['property_type'].isin(top_props), 'Other')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Room type: median price bar
room_order = df.groupby('room_type')['price'].median().sort_values(ascending=False).index
room_colors = {'Entire home/apt': '#E53935', 'Hotel room': '#FB8C00', 'Private room': '#43A047', 'Shared room': '#1E88E5'}
medians_rt = df.groupby('room_type')['price'].median().reindex(room_order)
bars = axes[0].bar(room_order, medians_rt, color=[room_colors.get(r, 'grey') for r in room_order], alpha=0.85, width=0.55)
axes[0].set_title('Median Price by Room Type')
axes[0].set_ylabel('Median price per night (£)')
for bar, val in zip(bars, medians_rt):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, f'£{val:.0f}', ha='center', fontweight='bold')

# Top property types: horizontal bar sorted by median
prop_medians = (
    df[df['property_type'].isin(top_props)]
    .groupby('property_type')['price']
    .median()
    .sort_values()
)
axes[1].barh(prop_medians.index, prop_medians.values, color='steelblue', alpha=0.8, height=0.65)
axes[1].set_title('Median Price by Property Type (Top 10)')
axes[1].set_xlabel('Median price per night (£)')
for val, y in zip(prop_medians.values, range(len(prop_medians))):
    axes[1].text(val + 1, y, f'£{val:.0f}', va='center', fontsize=9)
axes[1].spines['left'].set_visible(False)
axes[1].tick_params(axis='y', length=0)

plt.tight_layout()
plt.show()

The granular property types reveal something the broad room-type field hides: **condos and guest suites command noticeably more than rental units** with the same room type classification. "Entire home" is a broad bucket — a basement flat and a Chelsea townhouse are both "entire home/apt" but live in different pricing worlds.

## 5. Capacity: Guests, Bedrooms, Beds

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

for ax, col, label, cap, color in [
    (axes[0], 'accommodates', 'Guests', 10, '#E53935'),
    (axes[1], 'bedrooms', 'Bedrooms', 6, '#FB8C00'),
    (axes[2], 'beds', 'Beds', 8, '#43A047'),
]:
    data = df_plot[df_plot[col] <= cap].groupby(col)['price']
    medians = data.median()
    counts = data.count()
    
    bars = ax.bar(medians.index.astype(str), medians.values, color=color, alpha=0.8, width=0.6)
    ax.set_title(f'Median Price vs {label}')
    ax.set_xlabel(f'Number of {label.lower()}')
    ax.set_ylabel('Median price (£)')
    
    # Sample size annotations
    for bar, n in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, 5, f'n={n//1000:.0f}k' if n >= 1000 else f'n={n}',
                ha='center', fontsize=7.5, color='white', fontweight='bold', va='bottom')

plt.suptitle('Price Scales with Capacity — But Non-Linearly', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

Price scales with capacity, but the curve flattens at the top — going from 2 to 3 bedrooms lifts the median price more (in % terms) than going from 5 to 6. Hosts seem to hit a ceiling around where group travel demand drops off. The first bedroom is worth the most.

In [ ]:
# Price per guest: does efficiency change with size?
df_plot['price_per_guest'] = df_plot['price'] / df_plot['accommodates'].clip(lower=1)

fig, ax = plt.subplots(figsize=(10, 5))
acc_data = df_plot[df_plot['accommodates'] <= 10].groupby('accommodates')['price_per_guest'].median()
ax.plot(acc_data.index, acc_data.values, marker='o', color='steelblue', linewidth=2.5, markersize=8)
ax.fill_between(acc_data.index, acc_data.values, alpha=0.15, color='steelblue')
ax.set_title('Price per Guest Drops as Group Size Grows')
ax.set_xlabel('Number of guests (accommodates)')
ax.set_ylabel('Median price per guest per night (£)')
ax.yaxis.set_major_formatter(mticker := plt.FuncFormatter(lambda x, _: f'£{x:.0f}'))
plt.tight_layout()
plt.show()

**Larger groups get a better per-person rate.** A solo traveler pays ~£80–100/night per head while a group of 8 splits down to ~£25–35/night each. This is the classic Airbnb value proposition vs. hotels for groups.

## 6. Review Scores: Do Ratings Predict Price?

In [ ]:
review_cols = [
    'review_scores_rating',
    'review_scores_accuracy',
    'review_scores_cleanliness',
    'review_scores_checkin',
    'review_scores_communication',
    'review_scores_location',
    'review_scores_value',
]

review_labels = ['Overall', 'Accuracy', 'Cleanliness', 'Check-in', 'Communication', 'Location', 'Value']

# Distribution of each review dimension
fig, axes = plt.subplots(2, 4, figsize=(18, 7))
axes = axes.flatten()

for i, (col, label) in enumerate(zip(review_cols, review_labels)):
    data = df_plot[col].dropna()
    axes[i].hist(data, bins=40, color='mediumpurple', edgecolor='white', linewidth=0.3)
    axes[i].axvline(data.median(), color='tomato', linestyle='--', lw=1.5, label=f'Median: {data.median():.2f}')
    axes[i].set_title(label)
    axes[i].set_xlabel('Score (out of 5)')
    axes[i].legend(fontsize=8)

axes[-1].set_visible(False)
plt.suptitle('Review Score Distributions — All Dimensions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation of each review dimension with price
corrs = {label: df_plot[col].corr(df_plot['price']) for col, label in zip(review_cols, review_labels)}
corr_series = pd.Series(corrs).sort_values()

fig, ax = plt.subplots(figsize=(9, 4))
colors = ['tomato' if v < 0 else 'steelblue' for v in corr_series.values]
ax.barh(corr_series.index, corr_series.values, color=colors, alpha=0.8, height=0.55)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Correlation of Review Scores with Price')
ax.set_xlabel('Pearson r')
ax.spines['left'].set_visible(False)
ax.tick_params(axis='y', length=0)
for val, y in zip(corr_series.values, range(len(corr_series))):
    ax.text(val + (0.002 if val >= 0 else -0.002), y, f'{val:.3f}', va='center', ha='left' if val >= 0 else 'right', fontsize=9)
plt.tight_layout()
plt.show()

Fascinating result: **review scores are essentially uncorrelated with price**, and in some cases slightly negatively correlated (better value score = cheaper listing). 

Why? Two opposing forces cancel each other out:
- Luxury listings have higher prices and tend to score higher on *location* and *accuracy*
- Budget listings that deliver on expectations score high on *value* and *communication*

The net effect is nearly zero correlation. **Don't use review scores to predict price — they're market-relative, not absolute quality measures.**

## 7. Superhost & Instant Booking: Is There a Premium?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, label, palette in [
    (axes[0], 'host_is_superhost', 'Superhost', {True: '#FF9800', False: '#90A4AE'}),
    (axes[1], 'instant_bookable', 'Instant Bookable', {True: '#43A047', False: '#90A4AE'}),
]:
    groups = df_plot.dropna(subset=[col]).groupby(col)['price']
    medians = groups.median()
    counts = groups.count()
    
    bars = ax.bar(
        [f"{label}\n(n={counts[k]:,})" for k in medians.index],
        medians.values,
        color=[palette[k] for k in medians.index],
        alpha=0.85, width=0.45
    )
    ax.set_title(f'Median Price: {label} vs Not')
    ax.set_ylabel('Median price per night (£)')
    for bar, val in zip(bars, medians.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'£{val:.0f}', ha='center', fontweight='bold')
    
    # Premium annotation
    vals = list(medians.values)
    pct_diff = (max(vals) - min(vals)) / min(vals) * 100
    ax.text(0.5, 0.9, f'+{pct_diff:.0f}% premium', transform=ax.transAxes, ha='center',
            fontsize=12, color='darkred', fontweight='bold')

plt.suptitle('Superhost & Instant Booking — Price Premiums', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

**Superhosts charge more, and instant-bookable listings are typically priced lower.** The superhost premium is real but modest (~10–15%) — it signals quality and lets hosts justify a slightly higher rate. Instant booking skews lower likely because it's a volume strategy: lower price → more bookings → no gaps.

## 8. Location: Inner vs. Outer London → Boroughs

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Inner vs Outer
zone_stats = df.groupby('london_zone')['price'].agg(['median', 'mean', 'count'])
print(zone_stats.round(0))

zone_colors = {'Inner London': '#E53935', 'Outer London': '#1E88E5'}
zone_med = df.groupby('london_zone')['price'].median()
bars = axes[0].bar(zone_med.index, zone_med.values, color=[zone_colors[z] for z in zone_med.index], alpha=0.85, width=0.45)
axes[0].set_title('Median Price: Inner vs. Outer London')
axes[0].set_ylabel('Median price per night (£)')
for bar, val in zip(bars, zone_med.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5, f'£{val:.0f}', ha='center', fontweight='bold', fontsize=13)

# Box plot by zone
inner = df_plot[df_plot['london_zone'] == 'Inner London']['price']
outer = df_plot[df_plot['london_zone'] == 'Outer London']['price']
bp = axes[1].boxplot([inner, outer], labels=['Inner London', 'Outer London'],
                     patch_artist=True, medianprops={'color': 'white', 'linewidth': 2.5})
bp['boxes'][0].set_facecolor('#E53935')
bp['boxes'][1].set_facecolor('#1E88E5')
for patch in bp['boxes']:
    patch.set_alpha(0.75)
axes[1].set_title('Price Distribution by Zone')
axes[1].set_ylabel('Price per night (£)')

plt.tight_layout()
plt.show()

In [ ]:
# All 32 boroughs sorted by median price
borough_med = (
    df[df['price'] <= 500]
    .groupby(['neighbourhood_cleansed', 'london_zone'])['price']
    .agg(median='median', count='count')
    .query('count >= 50')
    .sort_values('median', ascending=True)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(11, 14))
colors = [zone_colors[z] for z in borough_med['london_zone']]
bars = ax.barh(borough_med['neighbourhood_cleansed'], borough_med['median'], color=colors, alpha=0.85, height=0.7)

for bar, val in zip(bars, borough_med['median']):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f'£{val:.0f}', va='center', fontsize=9)

legend_handles = [mpatches.Patch(color=c, alpha=0.85, label=l) for l, c in zone_colors.items()]
ax.legend(handles=legend_handles, loc='lower right')
ax.set_title('Median Airbnb Price by London Borough', fontsize=14, fontweight='bold')
ax.set_xlabel('Median price per night (£)')
ax.spines['left'].set_visible(False)
ax.tick_params(axis='y', length=0)
plt.tight_layout()
plt.show()

**Kensington & Chelsea and Westminster are in a different pricing universe.** The spread from cheapest outer borough (~£60–70/night) to Kensington (~£200+/night) is 3×+ — for the same room type. Inner London boroughs dominate the top half almost entirely, with Outer London clustering at the bottom.

Note that Outer London isn't uniformly cheap — Richmond upon Thames sits squarely mid-tier (premium suburb with good transport links).

## 9. Geographic Price Maps

In [ ]:
# Interactive scatter map — dots colored by price
map_df = df[df['price'] <= 500].sample(min(12000, len(df)), random_state=42)

fig = px.scatter_mapbox(
    map_df,
    lat='latitude', lon='longitude',
    color='price',
    color_continuous_scale='Plasma',
    range_color=[0, 350],
    size='price',
    size_max=10,
    opacity=0.6,
    zoom=9.8,
    center={'lat': 51.5074, 'lon': -0.1278},
    mapbox_style='open-street-map',
    hover_data={'price': True, 'neighbourhood_cleansed': True, 'room_type': True,
                'bedrooms': True, 'latitude': False, 'longitude': False},
    title='London Airbnb Listings — Coloured by Nightly Price (£)',
    labels={'price': 'Price (£)', 'neighbourhood_cleansed': 'Borough'}
)
fig.update_layout(
    height=620,
    margin={'r': 0, 't': 40, 'l': 0, 'b': 0},
    coloraxis_colorbar=dict(title='Price (£)', tickprefix='£')
)
fig.show()

In [ ]:
# Density heatmap — where do expensive listings concentrate?
fig = px.density_mapbox(
    map_df,
    lat='latitude', lon='longitude',
    z='price',
    radius=12,
    zoom=9.8,
    center={'lat': 51.5074, 'lon': -0.1278},
    mapbox_style='open-street-map',
    color_continuous_scale='Hot',
    title='Price Concentration Map — Brighter = Expensive Listings Dense',
    labels={'z': 'Price (£)'}
)
fig.update_layout(height=620, margin={'r': 0, 't': 40, 'l': 0, 'b': 0})
fig.show()

The maps make the Central London premium visceral — a bright, dense cluster of expensive listings around Westminster, Kensington, Chelsea, Mayfair, and the City. The East End (Tower Hamlets, Newham) is notably cooler. The most striking pattern: **the River Thames acts as a rough price boundary** — north bank generally commands more than south bank at equivalent distances from the centre.

## 10. Amenities: Which Features Command a Premium?

In [ ]:
# Most common amenities
all_amenities = [a for sublist in df_plot['amenities_list'] for a in sublist]
amenity_counts = Counter(all_amenities)
top_amenities = [a for a, _ in amenity_counts.most_common(50)]

print('Top 20 amenities by frequency:')
for a, c in amenity_counts.most_common(20):
    pct = c / len(df_plot) * 100
    print(f'  {a:<40}  {c:>6,}  ({pct:.0f}%)')

In [ ]:
# For each amenity, compare median price of listings WITH vs WITHOUT
# Focus on amenities present in 5–80% of listings (common enough to be meaningful, rare enough to be differentiating)
focus_amenities = [
    'Wifi', 'Kitchen', 'Washer', 'Dryer', 'Dishwasher', 'Air conditioning',
    'Hot tub', 'Pool', 'Gym', 'EV charger', 'Bathtub',
    'Dedicated workspace', 'Parking', 'Long term stays allowed',
    'Self check-in', 'Pets allowed', 'Baby crib', 'Piano'
]

# Check exact names in the dataset first
available = {a for a in all_amenities}
focus_amenities = [a for a in focus_amenities if a in available]

premium_results = []
for amenity in focus_amenities:
    has_it = df_plot['amenities_list'].apply(lambda lst: amenity in lst)
    n_with = has_it.sum()
    if n_with < 100:
        continue
    med_with = df_plot[has_it]['price'].median()
    med_without = df_plot[~has_it]['price'].median()
    premium_pct = (med_with - med_without) / med_without * 100
    premium_results.append({
        'amenity': amenity,
        'premium_%': round(premium_pct, 1),
        'med_with': round(med_with, 0),
        'med_without': round(med_without, 0),
        'n_listings': n_with,
        'coverage_%': round(n_with / len(df_plot) * 100, 1)
    })

prem_df = pd.DataFrame(premium_results).sort_values('premium_%', ascending=False)
print(prem_df.to_string(index=False))

In [ ]:
prem_sorted = prem_df.sort_values('premium_%')

fig, ax = plt.subplots(figsize=(11, 6))
colors = ['tomato' if v < 0 else 'seagreen' for v in prem_sorted['premium_%']]
bars = ax.barh(prem_sorted['amenity'], prem_sorted['premium_%'], color=colors, alpha=0.85, height=0.65)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Amenity Price Premium — Listings With vs. Without (%)', fontsize=13, fontweight='bold')
ax.set_xlabel('Price premium (%)')
for bar, val, n in zip(bars, prem_sorted['premium_%'], prem_sorted['n_listings']):
    ax.text(val + (0.5 if val >= 0 else -0.5), bar.get_y() + bar.get_height()/2,
            f'{val:+.0f}%', va='center', ha='left' if val >= 0 else 'right', fontsize=9)
ax.spines['left'].set_visible(False)
ax.tick_params(axis='y', length=0)
plt.tight_layout()
plt.show()

A word of caution: **these amenity premiums are partially confounded by location**. Listings with pools and hot tubs tend to be in expensive areas anyway. That said, a few genuine premium signals emerge — air conditioning, dishwasher, and gym access appear in more upscale listings. Budget amenities like baby cribs and long-term stay options are associated with cheaper listings (different market segments).

## 11. Estimated Occupancy & Revenue

In [ ]:
df_occ = df[(df['price'] <= 500) & (df['estimated_occupancy_l365d'] > 0)].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Occupancy vs price scatter
sample = df_occ.sample(min(5000, len(df_occ)), random_state=42)
scatter = axes[0].scatter(
    sample['estimated_occupancy_l365d'],
    sample['price'],
    c=sample['price'],
    cmap='plasma', alpha=0.25, s=12
)
axes[0].set_title('Occupancy vs. Nightly Price')
axes[0].set_xlabel('Estimated occupied nights (last 365 days)')
axes[0].set_ylabel('Price per night (£)')
corr_occ = df_occ[['estimated_occupancy_l365d', 'price']].corr().iloc[0, 1]
axes[0].text(0.65, 0.9, f'r = {corr_occ:.3f}', transform=axes[0].transAxes, fontsize=11)

# Revenue distribution by zone
rev_data = {
    zone: df_occ[df_occ['london_zone'] == zone]['estimated_revenue_l365d'].clip(upper=50000)
    for zone in ['Inner London', 'Outer London']
}
bp = axes[1].boxplot(list(rev_data.values()), labels=list(rev_data.keys()),
                     patch_artist=True, medianprops={'color': 'white', 'lw': 2.5})
bp['boxes'][0].set_facecolor('#E53935')
bp['boxes'][1].set_facecolor('#1E88E5')
for patch in bp['boxes']:
    patch.set_alpha(0.75)
axes[1].set_title('Estimated Annual Revenue by Zone')
axes[1].set_ylabel('Estimated revenue last 365 days (£)')

plt.tight_layout()
plt.show()

print("Median estimated annual revenue:")
print(df_occ.groupby('london_zone')['estimated_revenue_l365d'].median().apply(lambda x: f'£{x:,.0f}').to_string())

**Higher price doesn't always mean more revenue** — expensive listings often sit empty longer. The r ≈ −0.1 correlation between price and occupancy confirms a mild trade-off. Inner London hosts make significantly more in total annual revenue, but there's huge variance within each zone.

## 12. What Actually Drives Price? — Random Forest

In [ ]:
df_model = df.copy()

# Encode categoricals
le = LabelEncoder()
df_model['room_type_enc'] = le.fit_transform(df_model['room_type'])
df_model['borough_enc'] = le.fit_transform(df_model['neighbourhood_cleansed'])
df_model['property_type_enc'] = le.fit_transform(df_model['property_type'].fillna('Unknown'))
df_model['superhost_enc'] = df_model['host_is_superhost'].map({True: 1, False: 0}).fillna(0)
df_model['instant_enc'] = df_model['instant_bookable'].map({True: 1, False: 0}).fillna(0)
df_model['zone_enc'] = (df_model['london_zone'] == 'Inner London').astype(int)

feature_cols = [
    'latitude', 'longitude',
    'borough_enc', 'zone_enc',
    'room_type_enc', 'property_type_enc',
    'accommodates', 'bedrooms', 'beds', 'bathrooms_num', 'n_amenities',
    'minimum_nights', 'availability_365',
    'number_of_reviews', 'reviews_per_month',
    'review_scores_rating', 'review_scores_cleanliness', 'review_scores_location',
    'calculated_host_listings_count', 'superhost_enc', 'instant_enc',
]

feature_labels = [
    'Latitude', 'Longitude',
    'Borough', 'Inner/Outer Zone',
    'Room Type', 'Property Type',
    'Accommodates', 'Bedrooms', 'Beds', 'Bathrooms', 'Num Amenities',
    'Min Nights', 'Availability',
    'Num Reviews', 'Reviews/Month',
    'Rating', 'Cleanliness Score', 'Location Score',
    'Host Listings Count', 'Superhost', 'Instant Bookable',
]

X = df_model[feature_cols].fillna(0)
y = np.log1p(df_model['price'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')

In [ ]:
rf = RandomForestRegressor(
    n_estimators=250,
    max_depth=14,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42
)

rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

mae = mean_absolute_error(np.expm1(y_test), np.expm1(y_pred))
r2 = r2_score(y_test, y_pred)

print(f'R²  = {r2:.3f}   (1.0 = perfect)')
print(f'MAE = £{mae:.2f} per night  (on original price scale)')

In [ ]:
importances = pd.Series(rf.feature_importances_, index=feature_labels).sort_values(ascending=True)

location_features = {'Latitude', 'Longitude', 'Borough', 'Inner/Outer Zone'}
capacity_features = {'Accommodates', 'Bedrooms', 'Beds', 'Bathrooms', 'Property Type'}
color_map = {
    **{f: '#E53935' for f in location_features},
    **{f: '#FB8C00' for f in capacity_features},
}
bar_colors = [color_map.get(f, '#1E88E5') for f in importances.index]

fig, ax = plt.subplots(figsize=(12, 8))
bars = ax.barh(importances.index, importances.values, color=bar_colors, alpha=0.85, height=0.7)
ax.set_xlabel('Feature Importance (mean decrease in impurity)')
ax.set_title('What Drives Airbnb Price in London? — Random Forest Feature Importance', fontsize=13, fontweight='bold')

for bar, val in zip(bars, importances.values):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)

legend_handles = [
    mpatches.Patch(color='#E53935', alpha=0.85, label='Location'),
    mpatches.Patch(color='#FB8C00', alpha=0.85, label='Capacity / Property'),
    mpatches.Patch(color='#1E88E5', alpha=0.85, label='Listing characteristics'),
]
ax.legend(handles=legend_handles, loc='lower right')
ax.spines['left'].set_visible(False)
ax.tick_params(axis='y', length=0)
plt.tight_layout()
plt.show()

loc_imp = importances[importances.index.isin(location_features)].sum()
cap_imp = importances[importances.index.isin(capacity_features)].sum()
other_imp = importances[~importances.index.isin(location_features | capacity_features)].sum()
print(f'\nLocation features combined:         {loc_imp:.1%}')
print(f'Capacity/property features combined: {cap_imp:.1%}')
print(f'All other features:                  {other_imp:.1%}')

In [ ]:
# Actual vs predicted
fig, ax = plt.subplots(figsize=(8, 6))
idx = np.random.choice(len(y_test), 3000, replace=False)
actual = np.expm1(y_test.values[idx])
predicted = np.expm1(y_pred[idx])

ax.scatter(actual, predicted, alpha=0.15, s=12, c=actual, cmap='plasma')
cap = 600
ax.plot([0, cap], [0, cap], 'r--', lw=1.5, label='Perfect')
ax.set_xlim(0, cap)
ax.set_ylim(0, cap)
ax.set_xlabel('Actual price (£)')
ax.set_ylabel('Predicted price (£)')
ax.set_title(f'Actual vs. Predicted — R² = {r2:.3f}, MAE = £{mae:.0f}', fontsize=12)
ax.legend()
plt.tight_layout()
plt.show()

## 13. Conclusions

### What drives Airbnb prices in London?

| Driver | Importance | Key finding |
|--------|-----------|-------------|
| **Location (lat/lon + borough)** | Highest | Accounts for the majority of price variance. Central London boroughs command 2–3× outer borough rates |
| **Accommodates / Bedrooms** | High | Each additional bedroom lifts price, but diminishing returns above 3 beds |
| **Room Type** | High | Entire home ≈ 3× private room |
| **Property Type** | Moderate | Condos and guest suites price above rental units |
| **Number of Amenities** | Moderate | More amenities correlate with higher price (proxy for overall listing quality) |
| **Superhost status** | Low–Moderate | ~10–15% premium, real but small |
| **Review scores** | Very low | Essentially no correlation with price — ratings are market-relative |
| **Instant bookable** | Low | Slightly negative — volume strategy for cheaper listings |

### Key takeaways

1. **Location is king.** Latitude and longitude alone are among the strongest individual predictors. The borough you're in sets your price floor — not your amenities or review scores.

2. **The Inner/Outer London divide is stark.** Inner London listings (15 boroughs) have a median ~40–60% higher than Outer London. Within Inner London, the W8 / SW1 / W1 corridor (Kensington, Westminster) is another tier above the rest.

3. **Capacity is the second-biggest driver.** Adding a bedroom raises price more reliably than any other non-location variable. Per-guest price drops sharply with group size — Airbnb's classic advantage over hotels for 4+ people.

4. **Review scores are almost useless for price prediction.** High ratings signal quality *within a price bracket*, not across brackets. This is expected — a 5-star £60/night studio and a 5-star £300/night penthouse are both excellent *relative to their price*.

5. **The model explains ~60–65% of price variance.** The remaining unexplained variance likely lives in:
   - Listing photos and presentation quality
   - Specific view (river view vs. car park)
   - Proximity to tube stations (not captured here)
   - Seasonal pricing (this is a September snapshot)
   - Host's own research and willingness to experiment

### What to try next
- Add **distance to nearest tube station** as a feature (strong signal)
- **Time-series analysis** across multiple scrape dates to see seasonal patterns
- A **geospatial clustering model** to define micro-market zones more precisely than borough boundaries
- **NLP on listing titles/descriptions** to extract quality signals (words like 'luxury', 'cosy', 'garden')